# Day 11: Three-Model Ensemble (Chemprop + LGBM + TabPFN)

**Goal:** Fix TabPFN API issues and build a proper 3-model ensemble

**Day 10 Results:**
- 2-model ensemble (Chemprop + LGBM): CV RAE = 0.5555, Test RAE = **0.62** ✅
- TabPFN failed due to API parameter issue

**Day 11 Strategy:**
1. Fix TabPFN initialization (remove incorrect parameter)
2. Train all 3 models with 5-fold CV
3. Test multiple ensemble combinations:
   - 2-model: Chemprop + LGBM (baseline from day 10)
   - 2-model: Chemprop + TabPFN
   - 2-model: LGBM + TabPFN
   - **3-model: Chemprop + LGBM + TabPFN**
4. Compare all ensembles and select best

**Target: <0.60 test RAE**

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path

# RDKit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

# ML
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
tqdm.pandas()
sns.set_style("whitegrid")

print("✅ Imports complete")

## 1. Load Data

In [ ]:
# Load full training data (using ALL 4139 molecules like day 10)
train_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TRAIN.csv")
test_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TEST_BLINDED.csv")

print(f"Full training set: {len(train_df)} molecules")
print(f"Test set: {len(test_df)} molecules")

# Use the counter-screen filtered dataset like day 10
train_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_counter-assay_TRAIN.csv")
print(f"\n✅ Using counter-screen passed molecules: {len(train_df)}")
print(f"\nTarget distribution:")
print(train_df['pEC50'].describe())

## 2. Setup Scaffold-Based CV

In [ ]:
def get_scaffold(smiles):
    """Get Bemis-Murcko scaffold."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return "invalid"
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaffold
    except:
        return "invalid"

def calculate_rae(y_true, y_pred):
    """Calculate Relative Absolute Error."""
    numerator = np.sum(np.abs(y_true - y_pred))
    denominator = np.sum(np.abs(y_true - np.mean(y_true)))
    return numerator / denominator if denominator != 0 else np.inf

print("Computing scaffolds...")
train_df["scaffold"] = [get_scaffold(s) for s in tqdm(train_df.SMILES)]
test_df["scaffold"] = [get_scaffold(s) for s in tqdm(test_df.SMILES)]

scaffold_to_group = {s: i for i, s in enumerate(train_df["scaffold"].unique())}
train_df["scaffold_group"] = train_df["scaffold"].map(scaffold_to_group)

# Validate SMILES
def is_valid_smiles(smi):
    return smi is not None and Chem.MolFromSmiles(smi) is not None

valid_mask = train_df['SMILES'].apply(is_valid_smiles)
nan_mask = ~np.isnan(train_df['pEC50'])
keep_mask = valid_mask & nan_mask

print(f"\nDropped {(~keep_mask).sum()} rows")
train_df = train_df[keep_mask].reset_index(drop=True)

y_train = train_df['pEC50'].values
scaffold_groups = train_df['scaffold_group'].values
group_kfold = GroupKFold(n_splits=5)

print(f"✅ Unique scaffolds: {len(scaffold_to_group)}")
print(f"✅ Final training size: {len(train_df)}")

## 3. Train Chemprop (Model 1)

In [ ]:
from chemprop import data as cpdata, models as cpmodels, featurizers
import chemprop.nn as cpnn
import lightning as pl
import torch

print("="*80)
print("MODEL 1: CHEMPROP (Message Passing Neural Network)")
print("="*80)

chemprop_oof_preds = np.zeros(len(train_df))
chemprop_fold_results = []

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(y_train, y_train, scaffold_groups), 1):
    print(f"\nFold {fold}/5...")
    
    train_smiles = train_df['SMILES'].iloc[train_idx].tolist()
    val_smiles = train_df['SMILES'].iloc[val_idx].tolist()
    train_targets = y_train[train_idx].reshape(-1, 1).tolist()
    val_targets = y_train[val_idx].reshape(-1, 1).tolist()
    
    featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
    
    train_data = cpdata.MoleculeDataset(
        [cpdata.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(train_smiles, train_targets)],
        featurizer=featurizer
    )
    val_data = cpdata.MoleculeDataset(
        [cpdata.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(val_smiles, val_targets)],
        featurizer=featurizer
    )
    
    train_loader = cpdata.build_dataloader(train_data, shuffle=True, batch_size=50)
    val_loader = cpdata.build_dataloader(val_data, shuffle=False, batch_size=50)
    
    mp = cpnn.BondMessagePassing()
    agg = cpnn.MeanAggregation()
    ffn = cpnn.RegressionFFN()
    model = cpmodels.MPNN(mp, agg, ffn)
    
    trainer = pl.Trainer(
        max_epochs=50,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False
    )
    trainer.fit(model, train_loader, val_loader)
    
    preds = trainer.predict(model, val_loader)
    y_pred = torch.cat(preds).numpy().flatten()
    chemprop_oof_preds[val_idx] = y_pred
    
    y_val = y_train[val_idx]
    rae = calculate_rae(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    chemprop_fold_results.append({'rae': rae, 'mae': mae, 'r2': r2})
    print(f"  RAE={rae:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

chemprop_cv_rae = np.mean([r['rae'] for r in chemprop_fold_results])
chemprop_cv_mae = np.mean([r['mae'] for r in chemprop_fold_results])

print(f"\n{'='*80}")
print(f"Chemprop CV: RAE={chemprop_cv_rae:.4f}, MAE={chemprop_cv_mae:.4f}")
print(f"{'='*80}")

## 4. Generate Features for LGBM and TabPFN

In [ ]:
import useful_rdkit_utils as uru

print("Generating features...")

# RDKit descriptors
print("1/2: RDKit descriptors...")
rdkit_desc = uru.RDKitDescriptors()
train_df["rdkit_desc"] = [rdkit_desc.calc_smiles(x) for x in tqdm(train_df.SMILES)]
test_df["rdkit_desc"] = [rdkit_desc.calc_smiles(x) for x in tqdm(test_df.SMILES)]

# Morgan fingerprints
print("\n2/2: Morgan fingerprints...")
def get_morgan_fp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(nbits)
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits))

train_df["morgan_fp"] = [get_morgan_fp(x) for x in tqdm(train_df.SMILES)]
test_df["morgan_fp"] = [get_morgan_fp(x) for x in tqdm(test_df.SMILES)]

# Prepare feature matrices
rdkit_train = np.stack(train_df.rdkit_desc.values)
rdkit_test = np.stack(test_df.rdkit_desc.values)

imputer = SimpleImputer(strategy='median')
rdkit_train = imputer.fit_transform(rdkit_train)
rdkit_test = imputer.transform(rdkit_test)

morgan_train = np.stack(train_df.morgan_fp.values)
morgan_test = np.stack(test_df.morgan_fp.values)

# Normalize
scaler_rdkit = StandardScaler()
scaler_morgan = StandardScaler()

rdkit_train_norm = scaler_rdkit.fit_transform(rdkit_train)
rdkit_test_norm = scaler_rdkit.transform(rdkit_test)

morgan_train_norm = scaler_morgan.fit_transform(morgan_train)
morgan_test_norm = scaler_morgan.transform(morgan_test)

# Combine features
X_train = np.hstack([rdkit_train_norm, morgan_train_norm])
X_test = np.hstack([rdkit_test_norm, morgan_test_norm])

print(f"\n✅ Feature matrix: {X_train.shape}")
print(f"✅ Test matrix: {X_test.shape}")

## 5. Train LGBM (Model 2)

In [ ]:
from lightgbm import LGBMRegressor

print("="*80)
print("MODEL 2: LGBM (Gradient Boosting on RDKit + Morgan)")
print("="*80)

lgbm_params = {
    'n_estimators': 3000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 7,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'verbose': -1
}

lgbm_oof_preds = np.zeros(len(train_df))
lgbm_fold_results = []

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_train, y_train, scaffold_groups), 1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    model = LGBMRegressor(**lgbm_params, random_state=fold)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val)
    lgbm_oof_preds[val_idx] = y_pred
    
    rae = calculate_rae(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    lgbm_fold_results.append({'rae': rae, 'mae': mae, 'r2': r2})
    print(f"Fold {fold}: RAE={rae:.4f}, MAE={mae:.4f}, R²={r2:.4f}")

lgbm_cv_rae = np.mean([r['rae'] for r in lgbm_fold_results])
lgbm_cv_mae = np.mean([r['mae'] for r in lgbm_fold_results])

print(f"\n{'='*80}")
print(f"LGBM CV: RAE={lgbm_cv_rae:.4f}, MAE={lgbm_cv_mae:.4f}")
print(f"{'='*80}")

## 6. Train TabPFN (Model 3) - FIXED API

**Key fix:** Removed the incorrect `N_ensemble_configurations` parameter that caused day 10 to fail.

In [ ]:
from tabpfn import TabPFNRegressor

print("="*80)
print("MODEL 3: TabPFN (Prior-Fitted Neural Network)")
print("="*80)
print("\n⚠️  TabPFN limits: ~1000 samples, ~100 features")
print("Strategy: Subsample training, use top 100 RDKit features\n")

tabpfn_oof_preds = np.zeros(len(train_df))
tabpfn_fold_results = []
TABPFN_SUCCESS = True

for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_train, y_train, scaffold_groups), 1):
    print(f"Fold {fold}/5...")
    
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    # TabPFN limits
    max_samples = 1000
    max_features = 100
    
    if len(X_tr) > max_samples:
        print(f"  Subsampling {len(X_tr)} -> {max_samples} samples")
        subsample_idx = np.random.choice(len(X_tr), max_samples, replace=False)
        X_tr = X_tr[subsample_idx]
        y_tr = y_tr[subsample_idx]
    
    # Use first 100 features (RDKit descriptors only, no Morgan FPs)
    X_tr_subset = X_tr[:, :max_features]
    X_val_subset = X_val[:, :max_features]
    
    try:
        # FIXED: Use default parameters, no N_ensemble_configurations
        model = TabPFNRegressor(device='cpu')
        model.fit(X_tr_subset, y_tr)
        y_pred = model.predict(X_val_subset)
        tabpfn_oof_preds[val_idx] = y_pred
        
        rae = calculate_rae(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        
        tabpfn_fold_results.append({'rae': rae, 'mae': mae, 'r2': r2})
        print(f"  ✅ RAE={rae:.4f}, MAE={mae:.4f}, R²={r2:.4f}")
        
    except Exception as e:
        print(f"  ❌ TabPFN failed: {e}")
        TABPFN_SUCCESS = False
        break

if TABPFN_SUCCESS and len(tabpfn_fold_results) == 5:
    tabpfn_cv_rae = np.mean([r['rae'] for r in tabpfn_fold_results])
    tabpfn_cv_mae = np.mean([r['mae'] for r in tabpfn_fold_results])
    
    print(f"\n{'='*80}")
    print(f"✅ TabPFN CV: RAE={tabpfn_cv_rae:.4f}, MAE={tabpfn_cv_mae:.4f}")
    print(f"{'='*80}")
else:
    print(f"\n{'='*80}")
    print("❌ TabPFN training failed - will proceed with 2-model ensemble only")
    print(f"{'='*80}")
    tabpfn_cv_rae = None
    tabpfn_cv_mae = None

## 7. Compare Individual Models

In [ ]:
print("="*80)
print("INDIVIDUAL MODEL COMPARISON")
print("="*80)

comparison_data = [
    {'Model': 'Chemprop', 'CV RAE': chemprop_cv_rae, 'CV MAE': chemprop_cv_mae},
    {'Model': 'LGBM', 'CV RAE': lgbm_cv_rae, 'CV MAE': lgbm_cv_mae},
]

if TABPFN_SUCCESS:
    comparison_data.append({'Model': 'TabPFN', 'CV RAE': tabpfn_cv_rae, 'CV MAE': tabpfn_cv_mae})

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("="*80)

## 8. Test All Ensemble Combinations

In [ ]:
from itertools import product

print("="*80)
print("ENSEMBLE WEIGHT OPTIMIZATION")
print("="*80)

best_ensemble_rae = float('inf')
best_ensemble_config = None
ensemble_results = []

# Test 2-model ensembles
print("\n1. Two-Model Ensembles:")
print("-" * 80)

# Chemprop + LGBM (baseline from day 10)
print("\na) Chemprop + LGBM:")
weights_2model = [(0.3, 0.7), (0.4, 0.6), (0.5, 0.5), (0.6, 0.4), (0.7, 0.3)]

for w1, w2 in weights_2model:
    preds = w1 * chemprop_oof_preds + w2 * lgbm_oof_preds
    rae = calculate_rae(y_train, preds)
    mae = mean_absolute_error(y_train, preds)
    
    ensemble_results.append({
        'Type': '2-model',
        'Models': 'Chemprop + LGBM',
        'Weights': f"({w1:.1f}, {w2:.1f})",
        'RAE': rae,
        'MAE': mae
    })
    
    print(f"  ({w1:.1f}, {w2:.1f}): RAE={rae:.4f}, MAE={mae:.4f}")
    
    if rae < best_ensemble_rae:
        best_ensemble_rae = rae
        best_ensemble_config = {'type': '2-model', 'models': ['chemprop', 'lgbm'], 'weights': [w1, w2]}

if TABPFN_SUCCESS:
    # Chemprop + TabPFN
    print("\nb) Chemprop + TabPFN:")
    for w1, w2 in weights_2model:
        preds = w1 * chemprop_oof_preds + w2 * tabpfn_oof_preds
        rae = calculate_rae(y_train, preds)
        mae = mean_absolute_error(y_train, preds)
        
        ensemble_results.append({
            'Type': '2-model',
            'Models': 'Chemprop + TabPFN',
            'Weights': f"({w1:.1f}, {w2:.1f})",
            'RAE': rae,
            'MAE': mae
        })
        
        print(f"  ({w1:.1f}, {w2:.1f}): RAE={rae:.4f}, MAE={mae:.4f}")
        
        if rae < best_ensemble_rae:
            best_ensemble_rae = rae
            best_ensemble_config = {'type': '2-model', 'models': ['chemprop', 'tabpfn'], 'weights': [w1, w2]}
    
    # LGBM + TabPFN
    print("\nc) LGBM + TabPFN:")
    for w1, w2 in weights_2model:
        preds = w1 * lgbm_oof_preds + w2 * tabpfn_oof_preds
        rae = calculate_rae(y_train, preds)
        mae = mean_absolute_error(y_train, preds)
        
        ensemble_results.append({
            'Type': '2-model',
            'Models': 'LGBM + TabPFN',
            'Weights': f"({w1:.1f}, {w2:.1f})",
            'RAE': rae,
            'MAE': mae
        })
        
        print(f"  ({w1:.1f}, {w2:.1f}): RAE={rae:.4f}, MAE={mae:.4f}")
        
        if rae < best_ensemble_rae:
            best_ensemble_rae = rae
            best_ensemble_config = {'type': '2-model', 'models': ['lgbm', 'tabpfn'], 'weights': [w1, w2]}
    
    # Test 3-model ensemble
    print("\n2. Three-Model Ensemble:")
    print("-" * 80)
    print("\nChemprop + LGBM + TabPFN:")
    
    # Test a grid of weights (coarse grid to keep it manageable)
    weights_3model = []
    for w1 in [0.2, 0.3, 0.4]:
        for w2 in [0.3, 0.4, 0.5]:
            w3 = 1.0 - w1 - w2
            if 0.2 <= w3 <= 0.5:  # Ensure reasonable weights
                weights_3model.append((w1, w2, w3))
    
    for w1, w2, w3 in weights_3model:
        preds = w1 * chemprop_oof_preds + w2 * lgbm_oof_preds + w3 * tabpfn_oof_preds
        rae = calculate_rae(y_train, preds)
        mae = mean_absolute_error(y_train, preds)
        
        ensemble_results.append({
            'Type': '3-model',
            'Models': 'Chemprop + LGBM + TabPFN',
            'Weights': f"({w1:.1f}, {w2:.1f}, {w3:.1f})",
            'RAE': rae,
            'MAE': mae
        })
        
        print(f"  ({w1:.1f}, {w2:.1f}, {w3:.1f}): RAE={rae:.4f}, MAE={mae:.4f}")
        
        if rae < best_ensemble_rae:
            best_ensemble_rae = rae
            best_ensemble_config = {'type': '3-model', 'models': ['chemprop', 'lgbm', 'tabpfn'], 'weights': [w1, w2, w3]}

print(f"\n{'='*80}")
print(f"✅ BEST ENSEMBLE:")
print(f"   Type: {best_ensemble_config['type']}")
print(f"   Models: {' + '.join(best_ensemble_config['models'])}")
print(f"   Weights: {best_ensemble_config['weights']}")
print(f"   CV RAE: {best_ensemble_rae:.4f}")
print(f"{'='*80}")

## 9. Visualize Ensemble Results

In [ ]:
ensemble_df = pd.DataFrame(ensemble_results)

# Sort by RAE
ensemble_df = ensemble_df.sort_values('RAE')

print("\nTop 10 Ensemble Configurations:")
print(ensemble_df.head(10).to_string(index=False))

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# RAE comparison
if TABPFN_SUCCESS:
    models = ['Chemprop\n(alone)', 'LGBM\n(alone)', 'TabPFN\n(alone)', 
              'Chemprop+\nLGBM', 'Best\nEnsemble']
    raes = [chemprop_cv_rae, lgbm_cv_rae, tabpfn_cv_rae,
            ensemble_df[ensemble_df['Models'] == 'Chemprop + LGBM']['RAE'].min(),
            best_ensemble_rae]
else:
    models = ['Chemprop\n(alone)', 'LGBM\n(alone)', 'Chemprop+\nLGBM']
    raes = [chemprop_cv_rae, lgbm_cv_rae, best_ensemble_rae]

colors = ['steelblue'] * (len(models) - 1) + ['coral']
ax1.bar(models, raes, color=colors, edgecolor='black')
ax1.set_ylabel('Cross-Validation RAE', fontsize=12)
ax1.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax1.axhline(0.62, color='red', linestyle='--', linewidth=2, label='Day 10 Test RAE (target)')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Ensemble type comparison
ensemble_type_rae = ensemble_df.groupby('Type')['RAE'].agg(['mean', 'min', 'max'])
x = np.arange(len(ensemble_type_rae))
ax2.bar(x, ensemble_type_rae['mean'], yerr=ensemble_type_rae['max'] - ensemble_type_rae['min'],
        color='teal', edgecolor='black', capsize=10)
ax2.set_xticks(x)
ax2.set_xticklabels(ensemble_type_rae.index)
ax2.set_ylabel('RAE', fontsize=12)
ax2.set_title('Ensemble Type Performance', fontsize=14, fontweight='bold')
ax2.axhline(0.62, color='red', linestyle='--', linewidth=2, label='Day 10 Test RAE')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Train Final Models and Generate Test Predictions

In [ ]:
print("Training final models on full training set...\n")

# 1. Chemprop
print("1/3: Chemprop...")
train_smiles_all = train_df['SMILES'].tolist()
train_targets_all = y_train.reshape(-1, 1).tolist()
test_smiles_all = test_df['SMILES'].tolist()

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
train_data_full = cpdata.MoleculeDataset(
    [cpdata.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(train_smiles_all, train_targets_all)],
    featurizer=featurizer
)
test_data_full = cpdata.MoleculeDataset(
    [cpdata.MoleculeDatapoint.from_smi(smi) for smi in test_smiles_all],
    featurizer=featurizer
)

train_loader_full = cpdata.build_dataloader(train_data_full, shuffle=True, batch_size=50)
test_loader_full = cpdata.build_dataloader(test_data_full, shuffle=False, batch_size=50)

mp = cpnn.BondMessagePassing()
agg = cpnn.MeanAggregation()
ffn = cpnn.RegressionFFN()
chemprop_final = cpmodels.MPNN(mp, agg, ffn)

trainer = pl.Trainer(max_epochs=50, enable_progress_bar=True, enable_model_summary=False, logger=False)
trainer.fit(chemprop_final, train_loader_full)
chemprop_test_preds = torch.cat(trainer.predict(chemprop_final, test_loader_full)).numpy().flatten()
print(f"✅ Chemprop: mean={chemprop_test_preds.mean():.3f}, std={chemprop_test_preds.std():.3f}")

# 2. LGBM
print("\n2/3: LGBM...")
lgbm_final = LGBMRegressor(**lgbm_params, random_state=42)
lgbm_final.fit(X_train, y_train)
lgbm_test_preds = lgbm_final.predict(X_test)
print(f"✅ LGBM: mean={lgbm_test_preds.mean():.3f}, std={lgbm_test_preds.std():.3f}")

# 3. TabPFN (if available)
if TABPFN_SUCCESS:
    print("\n3/3: TabPFN...")
    max_samples = 1000
    max_features = 100
    
    if len(X_train) > max_samples:
        subsample_idx = np.random.choice(len(X_train), max_samples, replace=False)
        X_train_subset = X_train[subsample_idx, :max_features]
        y_train_subset = y_train[subsample_idx]
    else:
        X_train_subset = X_train[:, :max_features]
        y_train_subset = y_train
    
    tabpfn_final = TabPFNRegressor(device='cpu')
    tabpfn_final.fit(X_train_subset, y_train_subset)
    tabpfn_test_preds = tabpfn_final.predict(X_test[:, :max_features])
    print(f"✅ TabPFN: mean={tabpfn_test_preds.mean():.3f}, std={tabpfn_test_preds.std():.3f}")

# Create ensemble predictions
print(f"\n{'='*80}")
print("FINAL ENSEMBLE PREDICTIONS")
print(f"{'='*80}")

if best_ensemble_config['type'] == '2-model':
    w1, w2 = best_ensemble_config['weights']
    if best_ensemble_config['models'] == ['chemprop', 'lgbm']:
        ensemble_test_preds = w1 * chemprop_test_preds + w2 * lgbm_test_preds
    elif best_ensemble_config['models'] == ['chemprop', 'tabpfn']:
        ensemble_test_preds = w1 * chemprop_test_preds + w2 * tabpfn_test_preds
    else:  # lgbm, tabpfn
        ensemble_test_preds = w1 * lgbm_test_preds + w2 * tabpfn_test_preds
else:  # 3-model
    w1, w2, w3 = best_ensemble_config['weights']
    ensemble_test_preds = w1 * chemprop_test_preds + w2 * lgbm_test_preds + w3 * tabpfn_test_preds

print(f"Ensemble: mean={ensemble_test_preds.mean():.3f}, std={ensemble_test_preds.std():.3f}")
print(f"Training mean: {y_train.mean():.3f}")

## 11. Visualize Test Predictions

In [ ]:
n_models = 3 if TABPFN_SUCCESS else 2
fig, axes = plt.subplots(1, n_models + 1, figsize=(5 * (n_models + 1), 4))

# Chemprop
axes[0].hist(chemprop_test_preds, bins=30, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(y_train.mean(), color='red', linestyle='--', linewidth=2, label='Train mean')
axes[0].set_xlabel('pEC50', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Chemprop', fontsize=12, fontweight='bold')
axes[0].legend()

# LGBM
axes[1].hist(lgbm_test_preds, bins=30, alpha=0.7, color='green', edgecolor='black')
axes[1].axvline(y_train.mean(), color='red', linestyle='--', linewidth=2, label='Train mean')
axes[1].set_xlabel('pEC50', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].set_title('LGBM', fontsize=12, fontweight='bold')
axes[1].legend()

if TABPFN_SUCCESS:
    # TabPFN
    axes[2].hist(tabpfn_test_preds, bins=30, alpha=0.7, color='purple', edgecolor='black')
    axes[2].axvline(y_train.mean(), color='red', linestyle='--', linewidth=2, label='Train mean')
    axes[2].set_xlabel('pEC50', fontsize=11)
    axes[2].set_ylabel('Count', fontsize=11)
    axes[2].set_title('TabPFN', fontsize=12, fontweight='bold')
    axes[2].legend()

# Ensemble
axes[-1].hist(ensemble_test_preds, bins=30, alpha=0.7, color='coral', edgecolor='black')
axes[-1].axvline(y_train.mean(), color='red', linestyle='--', linewidth=2, label='Train mean')
axes[-1].set_xlabel('pEC50', fontsize=11)
axes[-1].set_ylabel('Count', fontsize=11)
axes[-1].set_title(f'Ensemble ({best_ensemble_config["type"]})', fontsize=12, fontweight='bold')
axes[-1].legend()

plt.tight_layout()
plt.show()

## 12. Create and Validate Submission

In [ ]:
submission_df = pd.DataFrame({
    'SMILES': test_df['SMILES'],
    'Molecule Name': test_df['Molecule Name'],
    'pEC50': ensemble_test_preds
})

# Save submission
output_path = '../outputs/day11_three_model_ensemble_submission.csv'
submission_df.to_csv(output_path, index=False)

print(f"✅ Submission saved to: {output_path}")
print(f"Shape: {submission_df.shape}")
print(f"\nFirst 10 predictions:")
print(submission_df.head(10))

# Validate
import sys
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from validation.activity_validation import validate_activity_submission

expected_activity_ids = set(test_df["Molecule Name"])
is_valid, validation_errors = validate_activity_submission(
    Path(output_path),
    expected_ids=expected_activity_ids,
)

if is_valid:
    print("\n✅ Submission file is valid!")
else:
    print("\n❌ Submission file is invalid:")
    for msg in validation_errors:
        print(f" - {msg}")

## 13. Summary and Analysis

In [ ]:
print("="*80)
print("DAY 11 SUMMARY")
print("="*80)

print(f"\n📊 Individual Model Performance (CV):")
print(f"   Chemprop: {chemprop_cv_rae:.4f} RAE")
print(f"   LGBM:     {lgbm_cv_rae:.4f} RAE")
if TABPFN_SUCCESS:
    print(f"   TabPFN:   {tabpfn_cv_rae:.4f} RAE")

print(f"\n🎯 Best Ensemble Configuration:")
print(f"   Type: {best_ensemble_config['type']}")
print(f"   Models: {' + '.join(best_ensemble_config['models'])}")
print(f"   Weights: {best_ensemble_config['weights']}")
print(f"   CV RAE: {best_ensemble_rae:.4f}")

print(f"\n📈 Comparison to Day 10:")
day10_rae = 0.5555
improvement = ((day10_rae - best_ensemble_rae) / day10_rae) * 100
print(f"   Day 10 (Chemprop + LGBM): {day10_rae:.4f} RAE")
print(f"   Day 11 (Best Ensemble):   {best_ensemble_rae:.4f} RAE")
if improvement > 0:
    print(f"   ✅ Improvement: {improvement:.2f}%")
elif improvement < 0:
    print(f"   ⚠️  Slightly worse: {abs(improvement):.2f}%")
else:
    print(f"   → Same performance")

print(f"\n🎲 Key Insights:")
if TABPFN_SUCCESS:
    print(f"   ✅ TabPFN API fixed successfully")
    if best_ensemble_config['type'] == '3-model':
        print(f"   ✅ 3-model ensemble outperformed 2-model ensembles")
        print(f"   → Model diversity helps: GNN + Trees + TabPFN capture different patterns")
    else:
        print(f"   → 2-model ensemble still best - TabPFN didn't add much value")
        print(f"   → Possible reasons: sample limit (1000), feature limit (100)")
else:
    print(f"   ❌ TabPFN still failed - proceeding with 2-model ensemble")

print(f"\n📋 Next Steps for <0.60 Test RAE:")
print(f"   1. Hyperparameter tuning (especially Chemprop learning rate, depth)")
print(f"   2. SMILES augmentation for Chemprop training")
print(f"   3. Stacking meta-learner (Ridge/Lasso on OOF predictions)")
print(f"   4. Feature engineering (MACCS keys, atom pairs for LGBM)")
print("="*80)